## Install the required Libraries

In [1]:
! pip install ollama

## Define the language model and embedding model

In [2]:
import ollama

language_model = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest"
embeddings_model = "hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest"

## Load the data

In [4]:
datasets = []
with open("D:/AgenticAI/ollama-app/cat-facts.txt", "r") as f:
    datasets = f.readlines()
    print(f"Loaded {len(datasets)} cat facts")

Loaded 150 cat facts


## Initializing the vector_db

In [5]:
vector_db = []

## Defining the embedding function

In [6]:
def add_chunk_to_vector_db(chunk):
    embedding = ollama.embed(model=embeddings_model, input=chunk)['embeddings'][0]
    vector_db.append((chunk, embedding))

## Vectorisation and storing in vector_db

In [7]:
for i, chunk in enumerate(datasets):
    add_chunk_to_vector_db(chunk)
    print(f"Added {i+1}/{len(datasets)} chunks to vector database")

Added 1/150 chunks to vector database
Added 2/150 chunks to vector database
Added 3/150 chunks to vector database
Added 4/150 chunks to vector database
Added 5/150 chunks to vector database
Added 6/150 chunks to vector database
Added 7/150 chunks to vector database
Added 8/150 chunks to vector database
Added 9/150 chunks to vector database
Added 10/150 chunks to vector database
Added 11/150 chunks to vector database
Added 12/150 chunks to vector database
Added 13/150 chunks to vector database
Added 14/150 chunks to vector database
Added 15/150 chunks to vector database
Added 16/150 chunks to vector database
Added 17/150 chunks to vector database
Added 18/150 chunks to vector database
Added 19/150 chunks to vector database
Added 20/150 chunks to vector database
Added 21/150 chunks to vector database
Added 22/150 chunks to vector database
Added 23/150 chunks to vector database
Added 24/150 chunks to vector database
Added 25/150 chunks to vector database
Added 26/150 chunks to vector data

In [8]:
vector_db

[('On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.\n',
  [-0.03609412,
   -0.022022257,
   0.046288613,
   -0.07995774,
   0.036077365,
   -0.014886799,
   0.0778245,
   0.054345153,
   -0.013937934,
   -0.0021585738,
   -0.019277334,
   -0.0065506776,
   -0.057554554,
   0.013800884,
   -0.048231225,
   0.039891694,
   0.08796751,
   0.011429542,
   -0.032701142,
   -0.030435419,
   0.003829148,
   0.027955338,
   -0.025594188,
   0.0001115271,
   0.049154025,
   -0.016156875,
   -0.008820397,
   -0.0021448918,
   0.003940243,
   -0.013264699,
   0.038277607,
   -0.029793879,
   -0.03372232,
   0.007034487,
   0.026024625,
   -0.03608568,
   -0.010429396,
   -0.03612161,
   0.015195706,
   0.032245588,
   -0.033295915,
   -0.0153117,
   -0.019713184,
   0.01304308,
   -0.030575413,
   -0.013332067,
   -0.0018612732,
   -0.014753321,
   0.02748535,
   0.016604796,
   -0.03332862,
   0.00077122607,
   0.00

## Defining Similarity Method (Cosine Similarity)

In [9]:
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = sum(a ** 2 for a in vec1) ** 0.5
    magnitude2 = sum(b ** 2 for b in vec2) ** 0.5
    if magnitude1 == 0 or magnitude2 == 0:
        return 0
    return dot_product / (magnitude1 * magnitude2)

## Defining the retriver function

In [10]:
def retriver(query, top_k=3):
    query_embedding = ollama.embed(model=embeddings_model, input=query)['embeddings'][0]
    similarities = []
    for chunk, embedding in vector_db:
        sim = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [chunk for chunk, _ in similarities[:top_k]]

## User Query

In [11]:
query = "What is the average lifespan of a cat?"

In [12]:
result = retriver(query)
print("Top relevant chunks:")
for i, chunk in enumerate(result):
    print(f"{i+1}. {chunk.strip()}")

Top relevant chunks:
1. When well treated, a cat can live twenty or more years but the average life span of a domestic cat is 14 years.
2. The oldest cat on record was Crème Puff from Austin, Texas, who lived from 1967 to August 6, 2005, three days after her 38th birthday. A cat typically can live up to 20 years, which is equivalent to about 96 human years.
3. On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
